In this code I attempt to perform the Levinsohn-Petrin procedure to estimation production function using one-step GMM

In [120]:
# Import packages 
using Pkg 
using GLM
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using Random
using Distributions
using Optim
using Plots
using ShiftedArrays  # for lag function

My current understanding is that we try to estimate the labor coefficient as well in the GMM, so basically write the moment conditions as a function of all the parameters we want to estimate 

In [121]:
# Step 1: Prepping datasets 
# we also prep the dataset such that we can compute lag_phi given parameter guess 
dataset = CSV.read("op_lp_ready.csv", DataFrame)

dataset.v_capital_square = dataset.v_capital .* dataset.v_capital
dataset.v_material_square = dataset.v_material .* dataset.v_material
dataset.int_capital_material = dataset.v_capital .* dataset.v_material

sort!(dataset, [:firm_id, :year])
transform!(groupby(dataset, :firm_id), :v_capital => (x -> lag(x, 1)) => :lag_capital) #generate capital lag variable
# transform!(groupby(dataset, :firm_id), :predicted_phi => (x -> lag(x, 1)) => :lag_phi) # generate phi lag variable
transform!(groupby(dataset, :firm_id), :v_material => (x -> lag(x, 1)) => :lag_material) # generate phi lag variable

# dataset = dropmissing(dataset, [:lag_capital, :lag_material])

display(first(dataset, 10))

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,v_capital_square,v_material_square,int_capital_material,lag_capital,lag_material
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64?,Float64?
1,1,1997,15.2437,12.2967,13.6469,14.7629,27.4075,186.238,217.944,201.468,missing,missing
2,1,1998,15.4399,12.6583,13.8412,15.0051,20.8425,191.578,225.152,207.688,13.6469,14.7629
3,1,1999,15.3679,12.3587,13.9283,14.9568,25.9536,193.999,223.707,208.324,13.8412,15.0051
4,2,1997,14.8474,12.3572,13.6883,14.2698,43.5369,187.37,203.627,195.329,missing,missing
5,2,1998,14.7601,11.5891,13.6968,14.2355,41.273,187.603,202.649,194.981,13.6883,14.2698
6,2,1999,14.7038,12.6857,13.8871,14.0906,42.8179,192.852,198.545,195.678,13.6968,14.2355
7,3,1998,13.9124,12.1451,12.0143,13.4115,17.4865,144.344,179.869,161.13,missing,missing
8,3,1999,13.66,9.89273,12.0052,13.0285,17.4949,144.124,169.741,156.409,12.0143,13.4115
9,4,1995,17.6699,14.5626,17.1751,17.2448,71.6182,294.985,297.384,296.182,missing,missing


In [122]:
function fitted_poly(parameter_guess,
    dataset)
    
    poly_1 = parameter_guess[6] # constant term 
    beta_l = parameter_guess[1] # poly_2 supposedly
    poly_3 = parameter_guess[7] # k term
    poly_4 = parameter_guess[8] # k^2 term
    poly_5 = parameter_guess[9] # m term
    poly_6 = parameter_guess[10] # m^2 term
    poly_7 = parameter_guess[11] # interaction k*m term 

    fitted_production = (poly_1 .+ beta_l .* dataset.v_labor .+
                        poly_3 .* dataset.v_capital .+ poly_4 .* dataset.v_capital_square .+ 
                        poly_5 .* dataset.v_material .+ poly_6 .* dataset.v_material_square .+ 
                        poly_7 .* dataset.int_capital_material)

    return fitted_production 
end 


function markov(parameter_guess, 
    lag_phi, lag_capital, lag_material)
    
    beta_k = parameter_guess[2] 
    beta_m = parameter_guess[3]
    alpha_0 = parameter_guess[4]
    alpha_1 = parameter_guess[5]   

    markov_term = alpha_0 .+ alpha_1 .* (lag_phi - beta_k .* dataset.lag_capital - beta_m .* dataset.lag_material)
    return markov_term 
end 

function RHS_step2(parameter_guess,
    dataset)

    beta_l = parameter_guess[1] # poly_2 supposedly
    beta_k = parameter_guess[2] 
    beta_m = parameter_guess[3]
    alpha_0 = parameter_guess[4]
    alpha_1 = parameter_guess[5]   
    poly_1 = parameter_guess[6] # constant term 
    poly_3 = parameter_guess[7] # k term
    poly_4 = parameter_guess[8] # k^2 term
    poly_5 = parameter_guess[9] # m term
    poly_6 = parameter_guess[10] # m^2 term
    poly_7 = parameter_guess[11] # interaction k*m term 

    sort!(dataset, [:firm_id, :year])
    dataset.lag_phi = fitted_poly(parameter_guess, dataset)

    # we also account for labor variation in this calculation as well 
    RHS_value = ((beta_l .* dataset.v_labor) + (beta_k .* dataset.v_capital) .+ (beta_m .* dataset.v_material) .+
                markov(parameter_guess, dataset.lag_phi, dataset.lag_capital, dataset.lag_material))
    return RHS_value 
end 

RHS_step2 (generic function with 1 method)

In [ ]:
# needs some WORK to stack moment conditions together 
# in this one-step procedure, we evaluate everything together 
function objective_function(parameter_guess, 
    dataset, matrix = I)

    beta_l = parameter_guess[1] # poly_2 supposedly
    beta_k = parameter_guess[2] 
    beta_m = parameter_guess[3]
    alpha_0 = parameter_guess[4]
    alpha_1 = parameter_guess[5]   
    poly_1 = parameter_guess[6] # constant term 
    poly_3 = parameter_guess[7] # k term
    poly_4 = parameter_guess[8] # k^2 term
    poly_5 = parameter_guess[9] # m term
    poly_6 = parameter_guess[10] # m^2 term
    poly_7 = parameter_guess[11] # interaction k*m term 

    RHS = RHS_step2(parameter_guess, dataset)
    sort!(dataset, [:firm_id, :year]) # sort the dataset to correct order

    eta = dataset.v_production .- fitted_poly(parameter_guess, dataset) # true production data minus what predicted from polynomial terms

    instrument_eta = Matrix(hcat(ones(length(dataset.v_capital)), dataset.v_capital, dataset.v_material, dataset.v_labor))  # Nx4 matrix (1, k, m, l)
    g_eta = (transpose(eta) * instrument_eta) ./ length(eta)  

    epsilon = dataset.v_production .- RHS # the production data minus what predicted from parameter guess
    instrument_epsilon = Matrix(hcat(ones(length(dataset.v_capital)), dataset.v_capital, dataset.lag_capital, dataset.lag_material))  # Nx4 matrix: (1, k, lag_k, lag_m)
    g_epsilon = (transpose(epsilon) * instrument_epsilon) ./ length(epsilon)

    moment_vec = vcat(g_eta[:], g_epsilon[:])  # 8x1 vector
    loss = moment_vec' * matrix * moment_vec  # equivalent to g' * I * g, as we use identity matrix for now
    return loss
end 

objective_function (generic function with 3 methods)

In [124]:
# function to run the GMM procedure 
dataset = dropmissing(dataset, [:lag_capital, :lag_material])

function GMM_onestep_main(
    initial_guess, 
    dataset
)
    # print out initial loss value
    println("Initial loss value: ", objective_function(initial_guess,dataset))

    result = optimize(parameter_guess -> objective_function(
            parameter_guess,
            dataset),
        initial_guess,
        NelderMead()
    )
    min_loss = Optim.minimum(result)   # minimum loss value
    println("Minimum loss value: ", min_loss)

    parameter_estimated = Optim.minimizer(result)
    println("Full estimated parameters:", parameter_estimated)

    println("Estimated labor coefficient [1]: ", parameter_estimated[1])
    println("Estimated capital coefficient [2]: ", parameter_estimated[2])
    println("Estimated material coefficient [3]: ", parameter_estimated[3])
    poly_terms = parameter_estimated[6:end]
    println("Estimated polynomial terms: ", poly_terms)
    return parameter_estimated
end 

parameter_estimated = GMM_onestep_main(ones(11), dataset)
# calculating mean productivity values by predicting production 

println("Mean productivity values: ", mean(fitted_poly(parameter_estimated,dataset)))

# A QUESTION STILL: HOW CAN WE HANDLE FIRST-STAGE POLYNOMIAL ESTIMATION WITHOUT DROPPING FIRST OBSERVATIONS?

Initial loss value: 4.7771666707968825e8
Minimum loss value: 0.3345671868354506
Full estimated parameters:[0.17339777145977006, 2.963531679724271, 4.690727053616333, -0.3906320503638907, 1.0516332963009203, 1.1240778580092208, 1.2272453512465418, -0.4087367613673608, 1.484103417113104, -0.31259472812166167, 0.5613745682383054]
Estimated labor coefficient [1]: 0.17339777145977006
Estimated capital coefficient [2]: 2.963531679724271
Estimated material coefficient [3]: 4.690727053616333
Estimated polynomial terms: [1.1240778580092208, 1.2272453512465418, -0.4087367613673608, 1.484103417113104, -0.31259472812166167, 0.5613745682383054]
Mean productivity values: 14.060644487579358
